In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Tuple, Optional, List, Set
import csv
import json
import time
import warnings

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ─── Paths ──────────────────────────────────────────────────────────────────
BASE = Path(".")
GAMES_LIVE2  = BASE / "data" / "games_live2"
PLAYERS_LIVE = BASE / "data" / "players_live"
PREDICTIONS  = BASE / "data" / "games_predictions.csv"
EVENTS_JSON  = BASE / "data" / "game_events.json"
OUTPUT_PATH  = BASE / "data" / "posterior_training_set3.csv"

TOTAL_GAME_SEC = 48 * 60  # 2880

# ─── Adaptive sampling ───────────────────────────────────────────────────────
# Power-law warping: frac^POWER maps game-time to bin-space.
#   power < 1  → more bins late-game (higher resolution when it matters)
#   power = 1  → uniform bins
# 0.7 gives a smooth ~3:1 late-vs-early resolution ratio.
ADAPTIVE_POWER  = 0.7
N_SAMPLE_BINS   = 160

# ─── Rolling-feature window ────────────────────────────────────────────────
ROLL_WINDOW_SEC = 120

# ─── Event-type ID sets ────────────────────────────────────────────────────
FT_IDS:    Set[int] = {97, 98, 99, 100, 101, 102, 103, 104, 105, 106,
                        107, 108, 157, 165, 166}
ALL_SHOT_IDS: Set[int] = set(range(91, 154)) | {282}
FG_IDS:    Set[int] = ALL_SHOT_IDS - FT_IDS
TOV_IDS:   Set[int] = set(range(62, 79)) | {84, 86, 87, 88, 89, 90, 478, 592}
FOUL_IDS:  Set[int] = {22, 24, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41,
                        42, 43, 44, 45, 47, 48, 257}
REB_IDS:   Set[int] = {155, 156}
DEF_REB, OFF_REB = 155, 156
TIMEOUT_IDS: Set[int] = {16, 17}

STATE_CHANGE: Set[int] = ALL_SHOT_IDS | FT_IDS | TOV_IDS | FOUL_IDS | REB_IDS | TIMEOUT_IDS
EXCLUDE_IDS:  Set[int] = {0, 402, 411, 412}

# ─── PBP columns ───────────────────────────────────────────────────────────
PBP_USECOLS = [
    "sequence_number", "type_id", "type_text", "text",
    "away_score", "home_score", "period_number",
    "clock_minutes", "clock_seconds",
    "scoring_play", "score_value",
    "end_game_seconds_remaining", "end_quarter_seconds_remaining",
    "season", "team_id",
    "away_team_id", "home_team_id", "game_date",
]

with open(EVENTS_JSON) as _f:
    EVENT_MAP: Dict[str, str] = json.load(_f)

print("Constants loaded")
print(f"  Adaptive sampling: power={ADAPTIVE_POWER}, bins={N_SAMPLE_BINS}")
print(f"  State-changing event types: {len(STATE_CHANGE)}")
print(f"  Paths OK: GAMES_LIVE2={GAMES_LIVE2.exists()}, "
      f"PREDICTIONS={PREDICTIONS.exists()}")

Constants loaded
  Adaptive sampling: power=0.7, bins=160
  State-changing event types: 115
  Paths OK: GAMES_LIVE2=True, PREDICTIONS=True


In [2]:
def season_from_date(dt: str) -> int:
    y, m = int(dt[:4]), int(dt[5:7])
    return y + 1 if m >= 7 else y


def build_pbp_index(root: Path) -> Dict[Tuple[str, str, str], List[Path]]:
    idx: Dict[Tuple[str, str, str], List[Path]] = {}
    for sdir in sorted(root.iterdir()):
        if not sdir.is_dir():
            continue
        s = sdir.name
        for f in sdir.iterdir():
            if f.suffix != ".csv":
                continue
            parts = f.stem.split("_")
            if len(parts) < 3:
                continue
            idx.setdefault((s, parts[1], parts[2]), []).append(f)
    return idx


def resolve_pbp(
    dt: str, away: str, home: str,
    idx: Dict[Tuple[str, str, str], List[Path]],
) -> Optional[Path]:
    season = str(season_from_date(dt))
    cands = idx.get((season, away, home), [])
    if not cands:
        return None
    if len(cands) == 1:
        return cands[0]
    for c in cands:
        try:
            with open(c) as fh:
                row = next(csv.DictReader(fh))
                if row.get("game_date", "").split("T")[0] == dt:
                    return c
        except (StopIteration, KeyError):
            continue
    return None


t0 = time.time()
PBP_IDX = build_pbp_index(GAMES_LIVE2)
n_files = sum(len(v) for v in PBP_IDX.values())
n_seasons = len({k[0] for k in PBP_IDX})
print(f"PBP index built: {n_files:,} files across {n_seasons} seasons "
      f"({time.time() - t0:.1f}s)")

PBP index built: 20,357 files across 16 seasons (0.1s)


In [3]:
def vec_sec_remaining_game(pbp: pd.DataFrame) -> np.ndarray:
    sec = pd.to_numeric(pbp.get("end_game_seconds_remaining"), errors="coerce").values.astype(float)
    mask = np.isnan(sec)
    if mask.any():
        p  = pd.to_numeric(pbp.loc[mask, "period_number"], errors="coerce").fillna(4).astype(int).values
        cm = pd.to_numeric(pbp.loc[mask, "clock_minutes"],  errors="coerce").fillna(0).astype(float).values
        cs = pd.to_numeric(pbp.loc[mask, "clock_seconds"],  errors="coerce").fillna(0).astype(float).values
        sec[mask] = (4 - p) * 600.0 + cm * 60.0 + cs
    return np.maximum(sec, 0.0)


def vec_sec_remaining_period(pbp: pd.DataFrame) -> np.ndarray:
    sec = pd.to_numeric(pbp.get("end_quarter_seconds_remaining"), errors="coerce").values.astype(float)
    mask = np.isnan(sec)
    if mask.any():
        cm = pd.to_numeric(pbp.loc[mask, "clock_minutes"], errors="coerce").fillna(0).astype(float).values
        cs = pd.to_numeric(pbp.loc[mask, "clock_seconds"], errors="coerce").fillna(0).astype(float).values
        sec[mask] = cm * 60.0 + cs
    return np.maximum(sec, 0.0)


def rolling_sum_time(
    vals: np.ndarray,
    sec_rem: np.ndarray,
    window: float,
) -> np.ndarray:
    n = len(vals)
    out = np.empty(n, dtype=np.float64)
    running = 0.0
    left = 0
    for i in range(n):
        running += vals[i]
        while left < i and sec_rem[left] - sec_rem[i] > window:
            running -= vals[left]
            left += 1
        out[i] = running
    return out


def safe_div(num: np.ndarray, denom: np.ndarray, fill: float = 0.0) -> np.ndarray:
    """Element-wise division; returns `fill` where denom <= 0."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(denom > 0, num / denom, fill)


def adaptive_bin_id(
    sec_remaining: np.ndarray,
    total_sec: float = TOTAL_GAME_SEC,
    power: float = ADAPTIVE_POWER,
    n_bins: int = N_SAMPLE_BINS,
) -> np.ndarray:
    """Map sec_remaining to adaptive bin IDs via power-law warping.

    frac = sec_remaining / total_sec   (1.0 at tip-off, 0.0 at buzzer)
    warped = frac^power

    With power < 1 the mapping expands late-game time (more bins near 0),
    giving higher sampling density when the game is on the line.

    Resolution profile (power=0.7, 160 bins):
      Game start  (~2880s left): ~1 bin per 23s
      Halftime    (~1440s left): ~1 bin per 17s
      5 min left  ( ~300s left): ~1 bin per 12s
      1 min left  (  ~60s left): ~1 bin per  8s
    """
    frac = np.clip(sec_remaining / total_sec, 0.0, 1.0)
    warped = np.power(frac, power)
    return np.clip((warped * n_bins).astype(int), 0, n_bins - 1)


# ── Quick sanity: show the resolution profile ───────────────────────────
print("Feature helpers defined\n")
print("Adaptive bin resolution profile:")
for sec_left in [2880, 1440, 720, 300, 120, 60, 30, 10, 0]:
    bid = adaptive_bin_id(np.array([sec_left]))[0]
    print(f"  {sec_left:>5}s remaining  ->  bin {bid:>3}")

Feature helpers defined

Adaptive bin resolution profile:
   2880s remaining  ->  bin 159
   1440s remaining  ->  bin  98
    720s remaining  ->  bin  60
    300s remaining  ->  bin  32
    120s remaining  ->  bin  17
     60s remaining  ->  bin  10
     30s remaining  ->  bin   6
     10s remaining  ->  bin   3
      0s remaining  ->  bin   0


In [4]:
def process_game(
    game_id: str,
    game_date: str,
    home_team: str,
    away_team: str,
    prior_home_wp: float,
    pbp_path: Path,
) -> pd.DataFrame:

    # ── 1. Read & sort ──────────────────────────────────────────────────
    try:
        pbp = pd.read_csv(pbp_path, usecols=PBP_USECOLS)
    except ValueError:
        pbp = pd.read_csv(pbp_path)

    assert len(pbp) > 0, f"Empty PBP file: {pbp_path}"
    pbp.sort_values("sequence_number", inplace=True)
    pbp.reset_index(drop=True, inplace=True)
    n = len(pbp)

    # ── 2. Coerce numeric types ─────────────────────────────────────────
    NUM_COLS = [
        "away_score", "home_score", "type_id", "score_value",
        "period_number", "clock_minutes", "clock_seconds",
        "end_game_seconds_remaining", "end_quarter_seconds_remaining",
        "team_id", "away_team_id", "home_team_id",
    ]
    for c in NUM_COLS:
        if c in pbp.columns:
            pbp[c] = pd.to_numeric(pbp[c], errors="coerce")

    pbp["score_value"] = pbp["score_value"].fillna(0).astype(int)
    tids = pbp["type_id"].fillna(-1).astype(int).values

    has_text = "text" in pbp.columns
    if has_text:
        text_lower = pbp["text"].fillna("").astype(str).str.lower().values
    else:
        text_lower = np.full(n, "", dtype=object)

    # ── 3. Team IDs ─────────────────────────────────────────────────────
    htid_col = pbp["home_team_id"].dropna()
    atid_col = pbp["away_team_id"].dropna()
    assert not htid_col.empty, f"No home_team_id for game {game_id}"
    assert not atid_col.empty, f"No away_team_id for game {game_id}"
    HID = int(htid_col.iloc[0])
    AID = int(atid_col.iloc[0])
    season = int(pbp["season"].dropna().iloc[0]) if "season" in pbp.columns else None

    # ── 4. Final outcome ────────────────────────────────────────────────
    final_h = pbp["home_score"].max()
    final_a = pbp["away_score"].max()
    assert pd.notna(final_h) and pd.notna(final_a), \
        f"Cannot determine final score for game {game_id}"
    home_win = int(final_h > final_a)

    # ── 5. Vectorised base arrays ───────────────────────────────────────
    tmids  = pbp["team_id"].values
    ttexts = pbp["type_text"].fillna("").astype(str).values

    sec_g = vec_sec_remaining_game(pbp)
    sec_p = vec_sec_remaining_period(pbp)

    period_vals = pbp["period_number"].fillna(0).astype(int).values
    is_ot_flag = (period_vals > 4).astype(int)

    h_sc = pbp["home_score"].ffill().fillna(0).astype(int).values
    a_sc = pbp["away_score"].ffill().fillna(0).astype(int).values
    s_diff = h_sc - a_sc

    ihe = np.where(tmids == HID, 1.0,
                   np.where(tmids == AID, 0.0, np.nan))

    sp_raw = pbp["scoring_play"].astype(str).str.strip().str.lower()
    is_sc = sp_raw.isin(["true", "1"]).astype(int).values
    sv = pbp["score_value"].values

    # ── 6. Event-type boolean arrays ────────────────────────────────────
    _fg    = np.isin(tids, list(FG_IDS))
    _ft    = np.isin(tids, list(FT_IDS))
    _tov   = np.isin(tids, list(TOV_IDS))
    _foul  = np.isin(tids, list(FOUL_IDS))
    _dreb  = (tids == DEF_REB)
    _oreb  = (tids == OFF_REB)
    _timeout = np.isin(tids, list(TIMEOUT_IDS))

    is_home = (ihe == 1)
    is_away = (ihe == 0)

    h_fg   = is_home & _fg;       a_fg   = is_away & _fg
    h_ft   = is_home & _ft;       a_ft   = is_away & _ft
    h_tov  = is_home & _tov;      a_tov  = is_away & _tov
    h_foul = is_home & _foul;     a_foul = is_away & _foul
    h_oreb = is_home & _oreb;     a_oreb = is_away & _oreb
    h_dreb = is_home & _dreb;     a_dreb = is_away & _dreb
    h_to_call = is_home & _timeout; a_to_call = is_away & _timeout

    h_fgm  = h_fg & (is_sc == 1);   a_fgm  = a_fg & (is_sc == 1)
    h_ftm  = h_ft & (is_sc == 1);   a_ftm  = a_ft & (is_sc == 1)
    h_fg3m = h_fgm & (sv == 3);     a_fg3m = a_fgm & (sv == 3)

    # ── 6b. Text-based event detection ──────────────────────────────────
    if has_text:
        _has_assist = np.array(["assists)" in t for t in text_lower], dtype=bool)
        _has_steal  = np.array(["steals)" in t for t in text_lower], dtype=bool)
        _has_block  = np.array(["block" in t for t in text_lower], dtype=bool)

        h_ast = is_home & _has_assist
        a_ast = is_away & _has_assist
        h_stl = is_away & _tov & _has_steal
        a_stl = is_home & _tov & _has_steal

        _missed_fg = _fg & (is_sc == 0)
        h_blk = is_away & _missed_fg & _has_block
        a_blk = is_home & _missed_fg & _has_block
    else:
        h_ast = np.zeros(n, dtype=bool)
        a_ast = np.zeros(n, dtype=bool)
        h_stl = np.zeros(n, dtype=bool)
        a_stl = np.zeros(n, dtype=bool)
        h_blk = np.zeros(n, dtype=bool)
        a_blk = np.zeros(n, dtype=bool)

    # ── 7. Possession inference ─────────────────────────────────────────
    poss = np.full(n, np.nan)
    cur: float = np.nan
    for i in range(n):
        t = tids[i]
        tm = tmids[i]
        if t != -1 and not np.isnan(tm):
            tm_int = int(tm)
            if t in ALL_SHOT_IDS or t in FT_IDS:
                cur = float(tm_int)
            elif t in TOV_IDS or "Turnover" in ttexts[i]:
                cur = float(tm_int)
            elif t == DEF_REB or t == OFF_REB:
                cur = float(tm_int)
        poss[i] = cur

    ihp = np.where(np.isnan(poss), np.nan,
                   np.where(poss == HID, 1.0, 0.0))

    # ── 8. Rolling 120-s features ───────────────────────────────────────
    h_sc_vals = np.where(is_home & (is_sc == 1), sv.astype(float), 0.0)
    a_sc_vals = np.where(is_away & (is_sc == 1), sv.astype(float), 0.0)

    hp120  = rolling_sum_time(h_sc_vals, sec_g, ROLL_WINDOW_SEC)
    ap120  = rolling_sum_time(a_sc_vals, sec_g, ROLL_WINDOW_SEC)
    ht120  = rolling_sum_time(h_tov.astype(float), sec_g, ROLL_WINDOW_SEC)
    at120  = rolling_sum_time(a_tov.astype(float), sec_g, ROLL_WINDOW_SEC)

    h_fgm_r120 = rolling_sum_time(h_fgm.astype(float), sec_g, ROLL_WINDOW_SEC)
    a_fgm_r120 = rolling_sum_time(a_fgm.astype(float), sec_g, ROLL_WINDOW_SEC)
    h_fga_r120 = rolling_sum_time(h_fg.astype(float),  sec_g, ROLL_WINDOW_SEC)
    a_fga_r120 = rolling_sum_time(a_fg.astype(float),  sec_g, ROLL_WINDOW_SEC)

    h_fg_pct_r120 = safe_div(h_fgm_r120, h_fga_r120)
    a_fg_pct_r120 = safe_div(a_fgm_r120, a_fga_r120)
    net_pts_r120 = hp120 - ap120

    # ── 9. Cumulative stats ─────────────────────────────────────────────
    cum_h_fga = np.cumsum(h_fg.astype(int))
    cum_a_fga = np.cumsum(a_fg.astype(int))
    cum_h_fta = np.cumsum(h_ft.astype(int))
    cum_a_fta = np.cumsum(a_ft.astype(int))
    cum_h_or  = np.cumsum(h_oreb.astype(int))
    cum_a_or  = np.cumsum(a_oreb.astype(int))
    cum_h_dr  = np.cumsum(h_dreb.astype(int))
    cum_a_dr  = np.cumsum(a_dreb.astype(int))

    cum_h_fgm  = np.cumsum(h_fgm.astype(int))
    cum_a_fgm  = np.cumsum(a_fgm.astype(int))
    cum_h_ftm  = np.cumsum(h_ftm.astype(int))
    cum_a_ftm  = np.cumsum(a_ftm.astype(int))
    cum_h_fg3m = np.cumsum(h_fg3m.astype(int))
    cum_a_fg3m = np.cumsum(a_fg3m.astype(int))
    cum_h_tov  = np.cumsum(h_tov.astype(int))
    cum_a_tov  = np.cumsum(a_tov.astype(int))
    cum_h_ast  = np.cumsum(h_ast.astype(int))
    cum_a_ast  = np.cumsum(a_ast.astype(int))
    cum_h_stl  = np.cumsum(h_stl.astype(int))
    cum_a_stl  = np.cumsum(a_stl.astype(int))
    cum_h_blk  = np.cumsum(h_blk.astype(int))
    cum_a_blk  = np.cumsum(a_blk.astype(int))
    cum_h_pf   = np.cumsum(h_foul.astype(int))
    cum_a_pf   = np.cumsum(a_foul.astype(int))
    cum_h_to_called = np.cumsum(h_to_call.astype(int))
    cum_a_to_called = np.cumsum(a_to_call.astype(int))

    # ── 10. Efficiency features ─────────────────────────────────────────
    eff_h_fg_pct = safe_div(cum_h_fgm.astype(float), cum_h_fga.astype(float))
    eff_a_fg_pct = safe_div(cum_a_fgm.astype(float), cum_a_fga.astype(float))

    eff_h_ft_pct = safe_div(cum_h_ftm.astype(float), cum_h_fta.astype(float))
    eff_a_ft_pct = safe_div(cum_a_ftm.astype(float), cum_a_fta.astype(float))

    eff_h_efg = safe_div(cum_h_fgm + 0.5 * cum_h_fg3m, cum_h_fga.astype(float))
    eff_a_efg = safe_div(cum_a_fgm + 0.5 * cum_a_fg3m, cum_a_fga.astype(float))

    eff_h_ts = safe_div(h_sc.astype(float),
                        2.0 * (cum_h_fga + 0.44 * cum_h_fta).astype(float))
    eff_a_ts = safe_div(a_sc.astype(float),
                        2.0 * (cum_a_fga + 0.44 * cum_a_fta).astype(float))

    cum_h_poss = (cum_h_fga - cum_h_or + cum_h_tov + 0.44 * cum_h_fta).astype(float)
    cum_a_poss = (cum_a_fga - cum_a_or + cum_a_tov + 0.44 * cum_a_fta).astype(float)

    eff_h_off_rtg = safe_div(h_sc.astype(float), cum_h_poss) * 100.0
    eff_a_off_rtg = safe_div(a_sc.astype(float), cum_a_poss) * 100.0

    # ── 11. Assemble feature matrix ─────────────────────────────────────
    # _type_id is kept temporarily for filtering, then dropped.
    # event_type_id / is_scoring_play / score_value are intentionally
    # excluded as model features: they describe the transient event,
    # not the game state.  After adaptive binning, event_type_id is
    # just whatever happened last in each time bin — noisy and
    # uninformative.  The cumulative and rolling stats already capture
    # the actual game state the model needs.

    feat = pd.DataFrame({
        "game_id":              game_id,
        "season":               season,
        "game_date":            game_date,
        "play_index":           pbp["sequence_number"].values,
        "prior_home_wp":        prior_home_wp,

        "home_score":           h_sc,
        "away_score":           a_sc,
        "score_diff":           s_diff,

        "period_number":        period_vals,
        "sec_remaining_game":   sec_g,
        "sec_remaining_period": sec_p,
        "is_overtime":          is_ot_flag,

        "is_home_possession":   ihp,

        "home_points_last_120": hp120,
        "away_points_last_120": ap120,
        "home_tov_last_120":    ht120,
        "away_tov_last_120":    at120,
        "home_fgm_last_120":    h_fgm_r120,
        "away_fgm_last_120":    a_fgm_r120,
        "home_fga_last_120":    h_fga_r120,
        "away_fga_last_120":    a_fga_r120,
        "home_fg_pct_last_120": h_fg_pct_r120,
        "away_fg_pct_last_120": a_fg_pct_r120,
        "net_pts_last_120":     net_pts_r120,

        "home_fga_to_date":     cum_h_fga,
        "away_fga_to_date":     cum_a_fga,
        "home_fta_to_date":     cum_h_fta,
        "away_fta_to_date":     cum_a_fta,
        "home_oreb_to_date":    cum_h_or,
        "away_oreb_to_date":    cum_a_or,
        "home_dreb_to_date":    cum_h_dr,
        "away_dreb_to_date":    cum_a_dr,
        "home_fgm_to_date":     cum_h_fgm,
        "away_fgm_to_date":     cum_a_fgm,
        "home_ftm_to_date":     cum_h_ftm,
        "away_ftm_to_date":     cum_a_ftm,
        "home_fg3m_to_date":    cum_h_fg3m,
        "away_fg3m_to_date":    cum_a_fg3m,
        "home_tov_to_date":     cum_h_tov,
        "away_tov_to_date":     cum_a_tov,
        "home_ast_to_date":     cum_h_ast,
        "away_ast_to_date":     cum_a_ast,
        "home_stl_to_date":     cum_h_stl,
        "away_stl_to_date":     cum_a_stl,
        "home_blk_to_date":     cum_h_blk,
        "away_blk_to_date":     cum_a_blk,
        "home_pf_to_date":      cum_h_pf,
        "away_pf_to_date":      cum_a_pf,
        "home_timeouts_called": cum_h_to_called,
        "away_timeouts_called": cum_a_to_called,

        "home_fg_pct":          eff_h_fg_pct,
        "away_fg_pct":          eff_a_fg_pct,
        "home_ft_pct":          eff_h_ft_pct,
        "away_ft_pct":          eff_a_ft_pct,
        "home_efg_pct":         eff_h_efg,
        "away_efg_pct":         eff_a_efg,
        "home_ts_pct":          eff_h_ts,
        "away_ts_pct":          eff_a_ts,
        "home_poss_to_date":    cum_h_poss,
        "away_poss_to_date":    cum_a_poss,
        "home_off_rtg":         eff_h_off_rtg,
        "away_off_rtg":         eff_a_off_rtg,

        "home_win_final":       home_win,

        # internal — used for filtering, dropped before return
        "_type_id":             tids,
    })

    # ── 12. Filter meta-events, keep only state-changing ────────────────
    feat = feat[~feat["_type_id"].isin(EXCLUDE_IDS)].copy()
    feat = feat[feat["_type_id"].isin(STATE_CHANGE)].copy()

    # ── 13. Adaptive downsampling ───────────────────────────────────────
    # Regulation: power-law time bins (smooth decay, more resolution late)
    # Overtime:   keep all state-changing events (OT periods are short)
    is_ot_mask = feat["is_overtime"] == 1
    reg = feat[~is_ot_mask].copy()
    ot  = feat[is_ot_mask].copy()

    if not reg.empty:
        reg["_bin"] = adaptive_bin_id(reg["sec_remaining_game"].values)
        reg = reg.groupby("_bin", sort=False).last().reset_index(drop=True)

    out = pd.concat([reg, ot], ignore_index=True)
    out.sort_values("play_index", inplace=True)
    out.reset_index(drop=True, inplace=True)

    # Drop internal columns
    out.drop(columns=["_type_id", "_bin"], inplace=True, errors="ignore")

    # ── 14. Uniform sample weight ───────────────────────────────────────
    # No artificial re-weighting — we trade the full game, so every
    # snapshot matters equally.  The adaptive bin density already gives
    # a mild ~3:1 late-vs-early resolution boost.
    out["sample_weight"] = 1.0

    return out


print("process_game defined (v3 — adaptive sampling, OT flag, no event_type_id)")

process_game defined (v3 — adaptive sampling, OT flag, no event_type_id)


In [5]:
# ─── Main pipeline ──────────────────────────────────────────────────────────

predictions = pd.read_csv(PREDICTIONS, dtype={"game_id": str})
# Prior model writes the home win-prob as `prior_home_wp`; the rest of this
# notebook references it as `pred_prob`. Alias to bridge the schema gap.
if "pred_prob" not in predictions.columns and "prior_home_wp" in predictions.columns:
    predictions["pred_prob"] = predictions["prior_home_wp"]
print(f"Loaded {len(predictions):,} prediction entries")

TEST_SEASON: Optional[int] = None
if TEST_SEASON is not None:
    dates = pd.to_datetime(predictions["game_date"])
    predictions["_season"] = dates.dt.year.where(dates.dt.month < 7, dates.dt.year + 1)
    predictions = predictions[predictions["_season"] == TEST_SEASON].copy()
    predictions.drop(columns=["_season"], inplace=True)
    print(f"  TEST MODE: filtered to season {TEST_SEASON} -> {len(predictions):,} games")

print(f"  Date range: {predictions['game_date'].min()} -> {predictions['game_date'].max()}")

results: List[pd.DataFrame] = []
skipped_no_pbp = 0
skipped_error  = 0
errors_log: List[str] = []

t0 = time.time()
total = len(predictions)

for i, row in predictions.iterrows():
    gid  = str(row["game_id"])
    gdt  = str(row["game_date"])
    ht   = str(row["home_team"])
    at   = str(row["away_team"])
    pwp  = float(row["pred_prob"])

    pbp_path = resolve_pbp(gdt, at, ht, PBP_IDX)
    if pbp_path is None:
        skipped_no_pbp += 1
        continue

    try:
        df = process_game(gid, gdt, ht, at, pwp, pbp_path)
        results.append(df)
    except Exception as e:
        skipped_error += 1
        msg = f"ERROR game {gid} ({gdt} {at}@{ht}): {e}"
        errors_log.append(msg)
        if skipped_error <= 20:
            print(msg)

    done = len(results) + skipped_no_pbp + skipped_error
    if done % 2000 == 0:
        elapsed = time.time() - t0
        rate = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate if rate > 0 else 0
        print(f"  {done:>6,} / {total:,}  |  "
              f"{len(results):,} ok, {skipped_no_pbp} no-pbp, {skipped_error} err  |  "
              f"{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining")

elapsed = time.time() - t0
print(f"\nPipeline complete in {elapsed:.0f}s")
print(f"  Processed:        {len(results):,}")
print(f"  Skipped (no PBP): {skipped_no_pbp:,}")
print(f"  Skipped (error):  {skipped_error:,}")

if errors_log:
    print(f"\n--- First {min(20, len(errors_log))} errors ---")
    for e in errors_log[:20]:
        print(f"  {e}")

Loaded 17,816 prediction entries
  Date range: 2010-11-14 -> 2026-05-26
   4,000 / 17,816  |  1,554 ok, 2446 no-pbp, 0 err  |  10s elapsed, ~33s remaining
  14,000 / 17,816  |  5,728 ok, 8272 no-pbp, 0 err  |  37s elapsed, ~10s remaining

Pipeline complete in 47s
  Processed:        7,200
  Skipped (no PBP): 10,616
  Skipped (error):  0


In [6]:
# ─── Concatenate, verify, and save ──────────────────────────────────────────

if not results:
    raise RuntimeError("No games were processed!")

output = pd.concat(results, ignore_index=True)

print(f"Output shape: {output.shape[0]:,} rows x {output.shape[1]} columns")
print(f"Columns ({len(output.columns)}): {list(output.columns)}\n")

# ── Label & weight summary ──────────────────────────────────────────────
print("Label distribution (home_win_final):")
print(output["home_win_final"].value_counts().to_string())
print(f"  Home win rate: {output['home_win_final'].mean():.4f}\n")

print(f"sample_weight: all {output['sample_weight'].unique()} (uniform)\n")

# ── Time & size summary ─────────────────────────────────────────────────
print(f"sec_remaining_game range: "
      f"[{output['sec_remaining_game'].min():.0f}, "
      f"{output['sec_remaining_game'].max():.0f}]")
print(f"Unique games: {output['game_id'].nunique():,}")
rps = output.groupby('game_id').size()
print(f"Rows per game: min={rps.min()}, median={rps.median():.0f}, max={rps.max()}")
print(f"Seasons covered: {sorted(output['season'].dropna().unique().astype(int))}")

# ── Overtime summary ────────────────────────────────────────────────────
n_ot_rows = (output["is_overtime"] == 1).sum()
n_ot_games = output.loc[output["is_overtime"] == 1, "game_id"].nunique()
print(f"Overtime: {n_ot_rows:,} rows across {n_ot_games:,} games\n")

# ── New feature spot-checks ─────────────────────────────────────────────
print("=== Feature spot-checks ===")
check_cols = [
    "home_fg_pct", "home_efg_pct", "home_ts_pct", "home_off_rtg",
    "home_poss_to_date", "home_ast_to_date", "home_stl_to_date",
    "home_blk_to_date", "home_pf_to_date", "home_timeouts_called",
    "home_fg_pct_last_120", "net_pts_last_120", "is_overtime",
]
for c in check_cols:
    if c in output.columns:
        s = output[c]
        print(f"  {c:30s}  min={s.min():8.2f}  median={s.median():8.2f}  max={s.max():8.2f}")
    else:
        print(f"  {c:30s}  ** MISSING **")
print()

# ── Verify dropped columns ──────────────────────────────────────────────
for gone in ["event_type_id", "is_scoring_play", "score_value", "_type_id", "_bin"]:
    assert gone not in output.columns, f"{gone} should have been dropped!"
print("Confirmed: event_type_id / is_scoring_play / score_value not in output")

# ── Sanity checks ──────────────────────────────────────────────────────
assert output["home_win_final"].isin([0, 1]).all()
assert output["sec_remaining_game"].ge(0).all()
assert (output["home_fg_pct"].between(0, 1) | (output["home_fg_pct"] == 0)).all()
assert output["is_overtime"].isin([0, 1]).all()
print("Sanity checks passed\n")

# ── Sampling density profile ────────────────────────────────────────────
print("=== Sampling density by game phase ===")
reg = output[output["is_overtime"] == 0]
for lo, hi, label in [(2400, 2880, "Q1 early"), (1440, 2400, "Q1-Q2"),
                       (720, 1440, "Q3"), (300, 720, "Q4 early"),
                       (0, 300, "Q4 clutch")]:
    chunk = reg[(reg["sec_remaining_game"] >= lo) & (reg["sec_remaining_game"] < hi)]
    n_rows = len(chunk)
    n_games = chunk["game_id"].nunique()
    rps_mean = n_rows / max(n_games, 1)
    sec_span = hi - lo
    sec_per_row = sec_span / max(rps_mean, 1)
    print(f"  {label:12s} ({lo:>4}-{hi:>4}s): "
          f"{rps_mean:5.1f} rows/game  ~{sec_per_row:.1f}s between snapshots")
print()

# ── Save ────────────────────────────────────────────────────────────────
output.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(output):,} rows to {OUTPUT_PATH}")

output.head(5)

Output shape: 990,309 rows x 64 columns
Columns (64): ['game_id', 'season', 'game_date', 'play_index', 'prior_home_wp', 'home_score', 'away_score', 'score_diff', 'period_number', 'sec_remaining_game', 'sec_remaining_period', 'is_overtime', 'is_home_possession', 'home_points_last_120', 'away_points_last_120', 'home_tov_last_120', 'away_tov_last_120', 'home_fgm_last_120', 'away_fgm_last_120', 'home_fga_last_120', 'away_fga_last_120', 'home_fg_pct_last_120', 'away_fg_pct_last_120', 'net_pts_last_120', 'home_fga_to_date', 'away_fga_to_date', 'home_fta_to_date', 'away_fta_to_date', 'home_oreb_to_date', 'away_oreb_to_date', 'home_dreb_to_date', 'away_dreb_to_date', 'home_fgm_to_date', 'away_fgm_to_date', 'home_ftm_to_date', 'away_ftm_to_date', 'home_fg3m_to_date', 'away_fg3m_to_date', 'home_tov_to_date', 'away_tov_to_date', 'home_ast_to_date', 'away_ast_to_date', 'home_stl_to_date', 'away_stl_to_date', 'home_blk_to_date', 'away_blk_to_date', 'home_pf_to_date', 'away_pf_to_date', 'home_timeou

,game_id,season,game_date,play_index,prior_home_wp,home_score,away_score,score_diff,period_number,sec_remaining_game,...,home_efg_pct,away_efg_pct,home_ts_pct,away_ts_pct,home_poss_to_date,away_poss_to_date,home_off_rtg,away_off_rtg,home_win_final,sample_weight
0,301114001,2011,2010-11-14,4,0.738339,0,2,-2,1,2505.0,...,0.000000,1.00,0.000000,1.000000,1.00,1.00,0.000000,200.000000,1,1.0
1,301114001,2011,2010-11-14,6,0.738339,0,2,-2,1,2478.0,...,0.000000,1.00,0.000000,1.000000,1.00,1.00,0.000000,200.000000,1,1.0
2,301114001,2011,2010-11-14,10,0.738339,3,4,-1,1,2456.0,...,0.750000,1.00,0.750000,1.063830,2.00,2.88,150.000000,138.888889,1,1.0
3,301114001,2011,2010-11-14,14,0.738339,5,4,1,1,2437.0,...,0.750000,0.50,0.868056,0.694444,2.88,3.88,173.611111,103.092784,1,1.0
4,301114001,2011,2010-11-14,18,0.738339,7,8,-1,1,2407.0,...,0.833333,0.75,0.902062,0.819672,3.88,4.88,180.412371,163.934426,1,1.0


In [7]:
# ─── 7. Modeling — Imports & Config ─────────────────────────────────────────

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
from typing import Any

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, log_loss

try:
    from xgboost import XGBClassifier
    RUN_XGB = True
    print("xgboost available")
except ImportError:
    RUN_XGB = False
    print("xgboost NOT installed — skipping XGB model")

# ─── Config ─────────────────────────────────────────────────────────────────
OUT_DIR     = "outputs_v3"
RANDOM_SEED = 42
TOTAL_GAME_SECONDS = 2880
DATA_PATH   = "data/posterior_training_set3.csv"

print(f"Config: DATA_PATH={DATA_PATH}, OUT_DIR={OUT_DIR}, SEED={RANDOM_SEED}")

xgboost available
Config: DATA_PATH=data/posterior_training_set3.csv, OUT_DIR=outputs_v3, SEED=42


In [8]:
# ─── 8. Utility Functions ───────────────────────────────────────────────────

def set_seeds(seed: int = RANDOM_SEED) -> None:
    np.random.seed(seed)
    print(f"Random seed set to {seed}")


def ensure_out_dir(path: str = OUT_DIR) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


# ── Metrics ─────────────────────────────────────────────────────────────────

def weighted_log_loss(
    y_true: np.ndarray, y_prob: np.ndarray, w: np.ndarray, eps: float = 1e-15,
) -> float:
    p = np.clip(y_prob, eps, 1 - eps)
    ll = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    return float(np.average(ll, weights=w))


def weighted_brier(
    y_true: np.ndarray, y_prob: np.ndarray, w: np.ndarray,
) -> float:
    return float(np.average((y_prob - y_true) ** 2, weights=w))


def weighted_auc(
    y_true: np.ndarray, y_prob: np.ndarray, w: np.ndarray,
) -> float:
    return float(roc_auc_score(y_true, y_prob, sample_weight=w))


def expected_calibration_error(
    y_true: np.ndarray, y_prob: np.ndarray, w: np.ndarray, n_bins: int = 15,
) -> float:
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total_w = w.sum()
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if not mask.any():
            continue
        bw = w[mask].sum()
        mean_pred = np.average(y_prob[mask], weights=w[mask])
        mean_true = np.average(y_true[mask], weights=w[mask])
        ece += (bw / total_w) * abs(mean_pred - mean_true)
    return float(ece)


def compute_all_metrics(
    y_true: np.ndarray, y_prob: np.ndarray, w: np.ndarray, prefix: str = "",
) -> Dict[str, float]:
    ones = np.ones_like(w)
    m: Dict[str, float] = {}
    for suffix, wt in [("_w", w), ("_uw", ones)]:
        tag = prefix + suffix
        m[f"{tag}_logloss"] = weighted_log_loss(y_true, y_prob, wt)
        m[f"{tag}_brier"]   = weighted_brier(y_true, y_prob, wt)
        m[f"{tag}_ece"]     = expected_calibration_error(y_true, y_prob, wt)
        m[f"{tag}_auc"]     = weighted_auc(y_true, y_prob, wt)
    return m


# ── Plotting ────────────────────────────────────────────────────────────────

def plot_reliability_diagram(
    y_true: np.ndarray, y_prob: np.ndarray, w: np.ndarray,
    n_bins: int = 15, title: str = "Reliability Diagram",
    save_path: Optional[str] = None,
) -> None:
    bin_edges = np.linspace(0, 1, n_bins + 1)
    mean_pred, mean_true, bin_weight = [], [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() < 5:
            continue
        bw = w[mask]
        mean_pred.append(np.average(y_prob[mask], weights=bw))
        mean_true.append(np.average(y_true[mask], weights=bw))
        bin_weight.append(bw.sum())
    mean_pred = np.array(mean_pred)
    mean_true = np.array(mean_true)
    bin_weight = np.array(bin_weight)
    bin_weight = bin_weight / bin_weight.max()

    fig, ax1 = plt.subplots(figsize=(6, 5))
    ax1.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    ax1.plot(mean_pred, mean_true, "o-", color="steelblue", label="Model")
    ax1.set_xlabel("Mean predicted probability")
    ax1.set_ylabel("Fraction of positives")
    ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-0.02, 1.02)
    ax1.set_title(title); ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.bar(mean_pred, bin_weight, width=1/n_bins*0.8, alpha=0.25, color="grey")
    ax2.set_ylabel("Relative bin weight"); ax2.set_ylim(0, 2.5)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved plot → {save_path}")
    plt.close(fig)


set_seeds()
out_dir = ensure_out_dir()
print(f"Output directory: {out_dir.resolve()}")

Random seed set to 42
Output directory: /Users/danielyang/Desktop/Projects/kalshi-trader/nba/outputs_v3


In [9]:
# ─── 9. Load & Clean Data ───────────────────────────────────────────────────
t0 = time.time()
raw = pd.read_csv(DATA_PATH, dtype={"game_id": str})
print(f"Loaded {len(raw):,} rows in {time.time()-t0:.1f}s")

raw["game_date"] = pd.to_datetime(raw["game_date"], errors="coerce")
raw["season"] = raw["season"].astype(int)
raw["home_win_final"] = raw["home_win_final"].astype(int)
assert raw["home_win_final"].isin([0, 1]).all()

raw["sample_weight"] = pd.to_numeric(raw["sample_weight"], errors="coerce").fillna(1.0).clip(lower=1e-6)
raw["is_home_possession"] = raw["is_home_possession"].fillna(-1.0)

raw.sort_values(["game_id", "play_index"], inplace=True)
raw.reset_index(drop=True, inplace=True)

n_games = raw["game_id"].nunique()
seasons = sorted(raw["season"].unique())
print(f"  Rows:    {len(raw):,}")
print(f"  Games:   {n_games:,}")
print(f"  Seasons: {seasons[0]}–{seasons[-1]} ({len(seasons)} total)")
print(f"  Label balance: {raw['home_win_final'].mean():.4f}")
print(f"  Features: {len([c for c in raw.columns if c not in ['game_id','play_index','game_date','season','home_win_final','sample_weight']])} numeric")

Loaded 990,309 rows in 1.5s
  Rows:    990,309
  Games:   7,200
  Seasons: 2011–2026 (16 total)
  Label balance: 0.5623
  Features: 58 numeric


In [10]:
# ─── 10. Train / Val / Test Split ───────────────────────────────────────────
# Temporal split by season — identical logic to v1.

max_season = raw["season"].max()
n_seasons  = raw["season"].nunique()

if n_seasons >= 3:
    train_seasons = [s for s in seasons if s <= max_season - 2]
    val_season    = max_season - 1
    test_season   = max_season

    df_train = raw[raw["season"].isin(train_seasons)].copy()
    df_val   = raw[raw["season"] == val_season].copy()
    df_test  = raw[raw["season"] == test_season].copy()
    split_desc = (f"Temporal — Train: {train_seasons[0]}–{train_seasons[-1]}, "
                  f"Val: {val_season}, Test: {test_season}")
else:
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
    rest_idx, test_idx = next(gss.split(raw, groups=raw["game_id"]))
    rest = raw.iloc[rest_idx]
    df_test = raw.iloc[test_idx].copy()
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=RANDOM_SEED)
    train_idx, val_idx = next(gss2.split(rest, groups=rest["game_id"]))
    df_train = rest.iloc[train_idx].copy()
    df_val   = rest.iloc[val_idx].copy()
    split_desc = "GroupShuffleSplit (single-season fallback)"

# Verify no game leakage
train_gids = set(df_train["game_id"])
val_gids   = set(df_val["game_id"])
test_gids  = set(df_test["game_id"])
assert train_gids.isdisjoint(val_gids),  "LEAK: train ∩ val"
assert train_gids.isdisjoint(test_gids), "LEAK: train ∩ test"
assert val_gids.isdisjoint(test_gids),   "LEAK: val ∩ test"

print(f"Split: {split_desc}")
for name, df_ in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    ng = df_["game_id"].nunique()
    ss = sorted(df_["season"].unique())
    print(f"  {name:5s}: {len(df_):>10,} rows, {ng:>5,} games, seasons {ss[0]}–{ss[-1]}")
print("No game-ID leakage across splits")

Split: Temporal — Train: 2011–2024, Val: 2025, Test: 2026
  Train:    862,281 rows, 6,304 games, seasons 2011–2024
  Val  :     62,811 rows,   440 games, seasons 2025–2025
  Test :     65,217 rows,   456 games, seasons 2026–2026
No game-ID leakage across splits


In [11]:
# ─── 11. Feature Preparation ────────────────────────────────────────────────
#
# Key changes from v1:
#   - No categorical features at all (event_type_id dropped in pipeline)
#   - No OneHotEncoder / ColumnTransformer
#   - StandardScaler only used for Logistic Regression (trees don't need it)
#   - All features are numeric → XGBoost/HGB get raw numpy arrays directly

from scipy.stats import norm

TARGET_COL = "home_win_final"
WEIGHT_COL = "sample_weight"
ID_COLS    = ["game_id", "play_index", "game_date"]
META_COLS  = [TARGET_COL, WEIGHT_COL, "season"]
DROP_COLS  = ID_COLS + META_COLS

FEATURE_COLS: List[str] = []  # populated on first call

# NBA scoring rate: ~110 pts / 2880 sec / team → score-diff variance ≈ 0.076 / sec
NBA_SCORE_DIFF_VAR_PER_SEC = 2 * (110.0 / 2880.0)


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["log_sec_game"]   = np.log1p(df["sec_remaining_game"].clip(lower=0))
    df["log_sec_period"] = np.log1p(df["sec_remaining_period"].clip(lower=0))
    df["time_frac"]      = (df["sec_remaining_game"] / TOTAL_GAME_SECONDS).clip(0, 1)

    # Fixed: weight by ELAPSED fraction — leads matter MORE as game progresses
    df["score_time_ix"] = df["score_diff"] * (1.0 - df["time_frac"])

    # Effective seconds remaining (OT: game clock is 0, use period clock)
    eff_sec = np.where(df["is_overtime"] == 1,
                       df["sec_remaining_period"].values,
                       df["sec_remaining_game"].values).astype(float)
    eff_sec = np.maximum(eff_sec, 0.5)

    # Estimated possessions remaining (~24 sec per possession)
    df["est_poss_remaining"] = np.maximum(eff_sec / 24.0, 0.5)

    # Lead significance: score_diff normalised by remaining-game volatility
    df["lead_significance"] = df["score_diff"].values / np.sqrt(df["est_poss_remaining"].values)

    # Analytical win probability from random-walk model.
    # P(final_lead > 0 | current_lead = L, time = t) = Φ(L / σ√t)
    # This gives the model a strong analytical baseline:
    #   - Early + small lead  → ~0.5  (noise, rely on prior)
    #   - Late  + big lead    → ~1.0  (game over, converge)
    std_remaining = np.sqrt(NBA_SCORE_DIFF_VAR_PER_SEC * eff_sec)
    df["analytical_wp"] = norm.cdf(df["score_diff"].values / std_remaining)

    return df


def make_features(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (X, y, w) as numpy arrays. All features are numeric."""
    global FEATURE_COLS
    df = add_derived_features(df)
    y = df[TARGET_COL].values.astype(int)
    w = df[WEIGHT_COL].values.astype(float)
    X = df.drop(columns=DROP_COLS, errors="ignore")
    if not FEATURE_COLS:
        FEATURE_COLS = list(X.columns)
    X = X[FEATURE_COLS].values.astype(np.float32)
    return X, y, w


X_train, y_train, w_train = make_features(df_train)
X_val,   y_val,   w_val   = make_features(df_val)
X_test,  y_test,  w_test  = make_features(df_test)

print(f"Feature columns ({len(FEATURE_COLS)}):")
print(f"  {FEATURE_COLS}")
print(f"\nAll numeric — no categorical features, no preprocessor needed for trees")
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

# Sanity: show derived feature values at key game moments
print("\nDerived feature sanity check:")
for label, sec, sdiff in [("Tip-off (0-0)", 2880, 0),
                           ("Q1 9:36 (down 1)", 2716, -1),
                           ("Half (up 5)", 1440, 5),
                           ("Q4 5:00 (up 5)", 300, 5),
                           ("Q4 0:34 (up 25)", 34, 25),
                           ("Q4 0:01 (up 3)", 1, 3)]:
    tf = sec / TOTAL_GAME_SECONDS
    stx = sdiff * (1.0 - tf)
    epr = max(sec / 24.0, 0.5)
    lsig = sdiff / np.sqrt(epr)
    std_r = np.sqrt(NBA_SCORE_DIFF_VAR_PER_SEC * max(sec, 0.5))
    awp = norm.cdf(sdiff / std_r)
    print(f"  {label:30s}  score_time_ix={stx:+7.2f}  "
          f"lead_sig={lsig:+6.2f}  analytical_wp={awp:.6f}")

Feature columns (65):
  ['prior_home_wp', 'home_score', 'away_score', 'score_diff', 'period_number', 'sec_remaining_game', 'sec_remaining_period', 'is_overtime', 'is_home_possession', 'home_points_last_120', 'away_points_last_120', 'home_tov_last_120', 'away_tov_last_120', 'home_fgm_last_120', 'away_fgm_last_120', 'home_fga_last_120', 'away_fga_last_120', 'home_fg_pct_last_120', 'away_fg_pct_last_120', 'net_pts_last_120', 'home_fga_to_date', 'away_fga_to_date', 'home_fta_to_date', 'away_fta_to_date', 'home_oreb_to_date', 'away_oreb_to_date', 'home_dreb_to_date', 'away_dreb_to_date', 'home_fgm_to_date', 'away_fgm_to_date', 'home_ftm_to_date', 'away_ftm_to_date', 'home_fg3m_to_date', 'away_fg3m_to_date', 'home_tov_to_date', 'away_tov_to_date', 'home_ast_to_date', 'away_ast_to_date', 'home_stl_to_date', 'away_stl_to_date', 'home_blk_to_date', 'away_blk_to_date', 'home_pf_to_date', 'away_pf_to_date', 'home_timeouts_called', 'away_timeouts_called', 'home_fg_pct', 'away_fg_pct', 'home_ft_pct

In [12]:
# ─── 12. Model Training ─────────────────────────────────────────────────────
#
# Changes from v1:
#   - No MLP (can't use sample_weight, was undertrained anyway)
#   - XGBoost gets raw numeric arrays (no OneHotEncoder / preprocessor)
#   - HGB gets raw arrays (no categorical_features mask needed)
#   - StandardScaler only wraps LR
#   - Broader XGBoost search with regularization (subsample, colsample,
#     min_child_weight, reg_alpha, reg_lambda)

models_registry: Dict[str, Dict[str, Any]] = {}

# ── A) Baseline: prior_home_wp ──────────────────────────────────────────
print("=" * 60)
print("A) Baseline — prior_home_wp")
print("=" * 60)
prior_col_idx = FEATURE_COLS.index("prior_home_wp")
val_prior  = X_val[:, prior_col_idx].clip(1e-6, 1 - 1e-6)
test_prior = X_test[:, prior_col_idx].clip(1e-6, 1 - 1e-6)
models_registry["prior_baseline"] = {
    "model": None,
    "val_pred": val_prior,
    "test_pred": test_prior,
}
print(f"  Val logloss: {weighted_log_loss(y_val, val_prior, w_val):.5f}")


# ── B) Logistic Regression (StandardScaler + sweep over C) ─────────────
print("\n" + "=" * 60)
print("B) Logistic Regression — C sweep (with StandardScaler)")
print("=" * 60)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_va_sc = scaler.transform(X_val)
X_te_sc = scaler.transform(X_test)

best_lr, best_lr_ll, best_lr_C = None, 1e9, None
for C_val in [0.01, 0.1, 0.3, 1.0, 3.0]:
    lr = LogisticRegression(
        C=C_val, max_iter=500, solver="lbfgs",
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    lr.fit(X_tr_sc, y_train, sample_weight=w_train)
    vp = lr.predict_proba(X_va_sc)[:, 1]
    ll = weighted_log_loss(y_val, vp, w_val)
    print(f"  C={C_val:<5}  val logloss={ll:.5f}")
    if ll < best_lr_ll:
        best_lr, best_lr_ll, best_lr_C = lr, ll, C_val

print(f"  Best C={best_lr_C}")
models_registry["logistic_regression"] = {
    "model": best_lr,
    "val_pred": best_lr.predict_proba(X_va_sc)[:, 1],
    "test_pred": best_lr.predict_proba(X_te_sc)[:, 1],
}


# ── C) HistGradientBoosting (raw numeric arrays) ───────────────────────
print("\n" + "=" * 60)
print("C) HistGradientBoostingClassifier")
print("=" * 60)

hgb = HistGradientBoostingClassifier(
    max_iter=400,
    max_depth=6,
    learning_rate=0.05,
    min_samples_leaf=50,
    max_bins=255,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=RANDOM_SEED,
)
hgb.fit(X_train, y_train, sample_weight=w_train)
vp = hgb.predict_proba(X_val)[:, 1]
ll = weighted_log_loss(y_val, vp, w_val)
print(f"  n_iter used: {hgb.n_iter_}")
print(f"  Val logloss: {ll:.5f}")
models_registry["hist_gbt"] = {
    "model": hgb,
    "val_pred": vp,
    "test_pred": hgb.predict_proba(X_test)[:, 1],
}


# ── D) XGBoost — broader search with regularization ────────────────────
if RUN_XGB:
    print("\n" + "=" * 60)
    print("D) XGBoost — extended grid with regularization")
    print("=" * 60)

    best_xgb, best_xgb_ll = None, 1e9
    configs = [
        {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 800,
         "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5,
         "reg_alpha": 0.0, "reg_lambda": 1.0},

        {"max_depth": 6, "learning_rate": 0.03, "n_estimators": 1200,
         "subsample": 0.8, "colsample_bytree": 0.7, "min_child_weight": 10,
         "reg_alpha": 0.1, "reg_lambda": 1.0},

        {"max_depth": 7, "learning_rate": 0.05, "n_estimators": 800,
         "subsample": 0.85, "colsample_bytree": 0.8, "min_child_weight": 5,
         "reg_alpha": 0.0, "reg_lambda": 1.5},

        {"max_depth": 5, "learning_rate": 0.05, "n_estimators": 1000,
         "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 10,
         "reg_alpha": 0.05, "reg_lambda": 1.0},

        {"max_depth": 6, "learning_rate": 0.02, "n_estimators": 1500,
         "subsample": 0.75, "colsample_bytree": 0.7, "min_child_weight": 15,
         "reg_alpha": 0.1, "reg_lambda": 2.0},
    ]
    for i, cfg in enumerate(configs):
        xgb_clf = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            early_stopping_rounds=30,
            **cfg,
        )
        xgb_clf.fit(
            X_train, y_train,
            sample_weight=w_train,
            eval_set=[(X_val, y_val)],
            sample_weight_eval_set=[w_val],
            verbose=False,
        )
        vp = xgb_clf.predict_proba(X_val)[:, 1]
        ll = weighted_log_loss(y_val, vp, w_val)
        print(f"  cfg[{i}] d={cfg['max_depth']} lr={cfg['learning_rate']} "
              f"n={cfg['n_estimators']} sub={cfg['subsample']} "
              f"col={cfg['colsample_bytree']} mcw={cfg['min_child_weight']} "
              f"a={cfg['reg_alpha']} l={cfg['reg_lambda']}"
              f"  →  val={ll:.5f} (iter={xgb_clf.best_iteration})")
        if ll < best_xgb_ll:
            best_xgb, best_xgb_ll = xgb_clf, ll

    models_registry["xgboost"] = {
        "model": best_xgb,
        "val_pred": best_xgb.predict_proba(X_val)[:, 1],
        "test_pred": best_xgb.predict_proba(X_test)[:, 1],
    }
else:
    print("Skipping XGBoost (not installed)")

print(f"\nModels trained: {list(models_registry.keys())}")

A) Baseline — prior_home_wp
  Val logloss: 0.60614

B) Logistic Regression — C sweep (with StandardScaler)


/opt/anaconda3/envs/kalshi-trader/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  C=0.01   val logloss=0.43086


/opt/anaconda3/envs/kalshi-trader/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  C=0.1    val logloss=0.43039


...

  C=0.3    val logloss=0.43038


/opt/anaconda3/envs/kalshi-trader/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  C=1.0    val logloss=0.43039


/opt/anaconda3/envs/kalshi-trader/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  C=3.0    val logloss=0.43039
  Best C=0.3

C) HistGradientBoostingClassifier
  n_iter used: 400
  Val logloss: 0.43080

D) XGBoost — extended grid with regularization
  cfg[0] d=6 lr=0.05 n=800 sub=0.8 col=0.8 mcw=5 a=0.0 l=1.0  →  val=0.42798 (iter=102)
  cfg[1] d=6 lr=0.03 n=1200 sub=0.8 col=0.7 mcw=10 a=0.1 l=1.0  →  val=0.42841 (iter=190)
  cfg[2] d=7 lr=0.05 n=800 sub=0.85 col=0.8 mcw=5 a=0.0 l=1.5  →  val=0.42954 (iter=113)
  cfg[3] d=5 lr=0.05 n=1000 sub=0.8 col=0.8 mcw=10 a=0.05 l=1.0  →  val=0.42669 (iter=193)
  cfg[4] d=6 lr=0.02 n=1500 sub=0.75 col=0.7 mcw=15 a=0.1 l=2.0  →  val=0.42811 (iter=286)

Models trained: ['prior_baseline', 'logistic_regression', 'hist_gbt', 'xgboost']


In [13]:
# ─── 13. Evaluation Summary ─────────────────────────────────────────────────

rows = []
for name, reg in models_registry.items():
    val_m  = compute_all_metrics(y_val,  reg["val_pred"],  w_val,  prefix="val")
    test_m = compute_all_metrics(y_test, reg["test_pred"], w_test, prefix="test")
    rows.append({"model": name, **val_m, **test_m})

metrics_df = pd.DataFrame(rows).set_index("model")

# Since weights are uniform (all 1.0), _w and _uw are identical.
# Show the unweighted columns for clarity.
display_cols = [
    "val_uw_logloss", "val_uw_brier", "val_uw_ece", "val_uw_auc",
    "test_uw_logloss", "test_uw_brier", "test_uw_ece", "test_uw_auc",
]
print("Evaluation Summary (uniform weights):\n")
print(metrics_df[display_cols].round(5).to_string())

best_name = metrics_df["val_uw_logloss"].idxmin()
print(f"\nBest model (by val logloss): {best_name}")
print(f"  val  logloss = {metrics_df.loc[best_name, 'val_uw_logloss']:.5f}")
print(f"  test logloss = {metrics_df.loc[best_name, 'test_uw_logloss']:.5f}")

Evaluation Summary (uniform weights):

                     val_uw_logloss  val_uw_brier  val_uw_ece  val_uw_auc  test_uw_logloss  test_uw_brier  test_uw_ece  test_uw_auc
model                                                                                                                              
prior_baseline              0.60614       0.20941     0.06718     0.73681          0.59064        0.20236      0.07978      0.76280
logistic_regression         0.43038       0.14442     0.01493     0.87490          0.38480        0.12626      0.02395      0.90339
hist_gbt                    0.43080       0.14482     0.02245     0.87467          0.38790        0.12716      0.01601      0.90101
xgboost                     0.42669       0.14313     0.02075     0.87729          0.38438        0.12583      0.02182      0.90345

Best model (by val logloss): xgboost
  val  logloss = 0.42669
  test logloss = 0.38438


In [14]:
# ─── 14. Calibration (Platt Scaling) ────────────────────────────────────────
#
# v1 used IsotonicRegression which overfit on val (produced NaN logloss,
# worsened brier/ECE on test).  Platt scaling fits a 2-parameter logistic
# regression on the raw predictions — far fewer degrees of freedom,
# much more stable on small val sets.

best_reg = models_registry[best_name]
val_raw  = best_reg["val_pred"]
test_raw = best_reg["test_pred"]

# Platt scaling: fit LR(1 feature = raw_pred) on val set
platt = LogisticRegression(C=1e10, max_iter=500, solver="lbfgs")
platt.fit(val_raw.reshape(-1, 1), y_val, sample_weight=w_val)

val_cal  = platt.predict_proba(val_raw.reshape(-1, 1))[:, 1]
test_cal = platt.predict_proba(test_raw.reshape(-1, 1))[:, 1]

print(f"Platt scaling fitted: coef={platt.coef_[0][0]:.4f}, intercept={platt.intercept_[0]:.4f}")
print(f"  (coef~1 & intercept~0 means model is already well-calibrated)\n")

print("Test set — raw vs Platt-calibrated:\n")
raw_m = compute_all_metrics(y_test, test_raw, w_test, prefix="test_raw")
cal_m = compute_all_metrics(y_test, test_cal, w_test, prefix="test_cal")
for key in ["_uw_logloss", "_uw_brier", "_uw_ece", "_uw_auc"]:
    r = raw_m[f"test_raw{key}"]
    c = cal_m[f"test_cal{key}"]
    delta = c - r
    arrow = "^" if delta > 0 else "v"
    print(f"  {key[4:]:>10s}:  raw={r:.5f}   cal={c:.5f}   ({arrow} {abs(delta):.5f})")

# Reliability plots
plot_reliability_diagram(
    y_test, test_raw, w_test,
    title=f"Test — {best_name} (raw)",
    save_path=str(out_dir / "reliability_raw.png"),
)
plot_reliability_diagram(
    y_test, test_cal, w_test,
    title=f"Test — {best_name} (Platt calibrated)",
    save_path=str(out_dir / "reliability_platt.png"),
)
print("\nCalibration complete")

Platt scaling fitted: coef=5.6661, intercept=-2.9141
  (coef~1 & intercept~0 means model is already well-calibrated)

Test set — raw vs Platt-calibrated:

     logloss:  raw=0.38438   cal=0.39532   (^ 0.01094)
       brier:  raw=0.12583   cal=0.12642   (^ 0.00059)
         ece:  raw=0.02182   cal=0.03096   (^ 0.00915)
         auc:  raw=0.90345   cal=0.90345   (v 0.00000)
  Saved plot → outputs_v3/reliability_raw.png
  Saved plot → outputs_v3/reliability_platt.png

Calibration complete


In [15]:
# ─── 15. Save Artifacts ─────────────────────────────────────────────────────

best_model = best_reg["model"]
model_path = out_dir / "best_model.joblib"

if best_name == "xgboost" and best_model is not None:
    xgb_json_path = out_dir / "best_model_xgb.json"
    best_model.save_model(str(xgb_json_path))
    print(f"  XGBoost JSON → {xgb_json_path}")

if best_model is not None:
    joblib.dump(best_model, model_path)
    print(f"  Model artifact → {model_path}")
else:
    print("  Best model is the prior baseline — no artifact to save.")

# Calibrator (Platt scaler)
cal_path = out_dir / "calibrator_platt.joblib"
joblib.dump(platt, cal_path)
print(f"  Platt calibrator → {cal_path}")

# StandardScaler (needed only for LR inference)
joblib.dump(scaler, out_dir / "scaler.joblib")
print(f"  StandardScaler → {out_dir / 'scaler.joblib'}")

# Metrics JSON
summary = {
    "best_model": best_name,
    "timestamp": datetime.now().isoformat(),
    "data_path": DATA_PATH,
    "random_seed": RANDOM_SEED,
    "train_rows": len(df_train),
    "val_rows": len(df_val),
    "test_rows": len(df_test),
    "feature_columns": FEATURE_COLS,
    "n_features": len(FEATURE_COLS),
    "calibration": "platt_scaling",
    "val_metrics_raw": {k: round(v, 6) for k, v in
                        compute_all_metrics(y_val, val_raw, w_val, "val").items()},
    "test_metrics_raw": {k: round(v, 6) for k, v in
                         compute_all_metrics(y_test, test_raw, w_test, "test").items()},
    "test_metrics_calibrated": {k: round(v, 6) for k, v in
                                compute_all_metrics(y_test, test_cal, w_test, "test_cal").items()},
    "all_models_val_logloss": {
        name: round(float(metrics_df.loc[name, "val_uw_logloss"]), 6)
        for name in metrics_df.index
    },
}
json_path = out_dir / "metrics_summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"  Metrics JSON → {json_path}")

# Test predictions CSV
pred_df = pd.DataFrame({
    "game_id":         df_test["game_id"].values,
    "play_index":      df_test["play_index"].values,
    "y_true":          y_test,
    "pred_raw":        test_raw,
    "pred_calibrated": test_cal,
})
csv_path = out_dir / "test_predictions.csv"
pred_df.to_csv(csv_path, index=False)
print(f"  Test predictions → {csv_path} ({len(pred_df):,} rows)")

print("\nAll artifacts saved.")

  XGBoost JSON → outputs_v3/best_model_xgb.json
  Model artifact → outputs_v3/best_model.joblib
  Platt calibrator → outputs_v3/calibrator_platt.joblib
  StandardScaler → outputs_v3/scaler.joblib
  Metrics JSON → outputs_v3/metrics_summary.json
  Test predictions → outputs_v3/test_predictions.csv (65,217 rows)

All artifacts saved.


In [16]:
# ─── 16. Inference Utilities ────────────────────────────────────────────────

def load_artifacts_v3(
    out_dir: str = OUT_DIR,
) -> Tuple[Any, Any, Dict]:
    """Load saved model, Platt calibrator, and metadata from disk."""
    od = Path(out_dir)
    metadata = json.loads((od / "metrics_summary.json").read_text())
    calibrator_ = joblib.load(od / "calibrator_platt.joblib")

    model_name = metadata["best_model"]
    if model_name == "xgboost" and (od / "best_model_xgb.json").exists():
        from xgboost import XGBClassifier as XGB_
        m = XGB_()
        m.load_model(str(od / "best_model_xgb.json"))
    elif (od / "best_model.joblib").exists():
        m = joblib.load(od / "best_model.joblib")
    else:
        m = None
    return m, calibrator_, metadata


def predict_proba_v3(X: np.ndarray, model: Any = None) -> np.ndarray:
    """Produce raw P(home_win) from a feature matrix.
    All models accept raw numeric arrays (no preprocessor needed for trees).
    """
    if model is None:
        return X[:, FEATURE_COLS.index("prior_home_wp")].clip(1e-6, 1 - 1e-6)
    return model.predict_proba(X)[:, 1]


def predict_proba_calibrated_v3(
    X: np.ndarray, model: Any = None, calibrator_: Any = None,
) -> np.ndarray:
    """Produce Platt-calibrated P(home_win)."""
    raw = predict_proba_v3(X, model)
    if calibrator_ is None:
        return raw
    return calibrator_.predict_proba(raw.reshape(-1, 1))[:, 1]


# ── Demo ────────────────────────────────────────────────────────────────
print("Inference demo (5 test rows):\n")
loaded_model, loaded_cal, loaded_meta = load_artifacts_v3()
raw_p = predict_proba_v3(X_test[:5], loaded_model)
cal_p = predict_proba_calibrated_v3(X_test[:5], loaded_model, loaded_cal)

demo_out = pd.DataFrame({
    "game_id":    df_test["game_id"].values[:5],
    "play_index": df_test["play_index"].values[:5],
    "y_true":     y_test[:5],
    "pred_raw":   raw_p,
    "pred_cal":   cal_p,
})
print(demo_out.to_string(index=False))
print(f"\nBest model: {loaded_meta['best_model']}")
print(f"Features: {loaded_meta['n_features']}")
print("Inference utilities ready.")

Inference demo (5 test rows):

  game_id  play_index  y_true  pred_raw  pred_cal
401809240           7       1  0.721462  0.763822
401809240          11       1  0.721462  0.763822
401809240          20       1  0.751778  0.793397
401809240          24       1  0.731962  0.774386
401809240          28       1  0.713632  0.755725

Best model: xgboost
Features: 65
Inference utilities ready.


In [18]:
# ─── 17. Manual Inspection — Single Game PBP with Predictions ───────────────
#
# Pick any row from games_predictions.csv, process the full PBP,
# run the model at each downsampled snapshot, then forward-fill
# predictions onto every play in the original PBP.
#
# Change ROW_IDX to inspect a different game.

ROW_IDX = 25000

pred_row = predictions.iloc[ROW_IDX]
gid = str(pred_row["game_id"])
gdt = str(pred_row["game_date"])
ht  = str(pred_row["home_team"])
at  = str(pred_row["away_team"])
pwp = float(pred_row["pred_prob"])

print(f"{'='*70}")
print(f"ROW_IDX = {ROW_IDX}")
print(f"  game_id:    {gid}")
print(f"  game_date:  {gdt}")
print(f"  away_team:  {at}")
print(f"  home_team:  {ht}")
print(f"  prior (pred_prob): {pwp:.6f}")
print(f"{'='*70}\n")

pbp_path = resolve_pbp(gdt, at, ht, PBP_IDX)
assert pbp_path is not None, f"No PBP file found for {gdt} {at}@{ht}"
print(f"PBP file: {pbp_path}")

# Quick sanity: check team IDs from the PBP match the expected game
_pbp_check = pd.read_csv(pbp_path, nrows=5)
if "home_team_id" in _pbp_check.columns and "away_team_id" in _pbp_check.columns:
    _htid = _pbp_check["home_team_id"].dropna().iloc[0] if not _pbp_check["home_team_id"].dropna().empty else "?"
    _atid = _pbp_check["away_team_id"].dropna().iloc[0] if not _pbp_check["away_team_id"].dropna().empty else "?"
    print(f"  PBP home_team_id={_htid}, away_team_id={_atid}")
del _pbp_check

# 1. Process game through feature pipeline (adaptive-downsampled snapshots)
game_df = process_game(gid, gdt, ht, at, pwp, pbp_path)
print(f"\nSnapshots from process_game: {len(game_df)}")
print(f"  Final score: home={game_df['home_score'].iloc[-1]}, away={game_df['away_score'].iloc[-1]}")
print(f"  Home win: {game_df['home_win_final'].iloc[0]}")

# 2. Run model on each snapshot
game_enriched = add_derived_features(game_df)
X_game = game_enriched.drop(columns=DROP_COLS, errors="ignore")[FEATURE_COLS].values.astype(np.float32)

raw_pred = loaded_model.predict_proba(X_game)[:, 1]
cal_pred = loaded_cal.predict_proba(raw_pred.reshape(-1, 1))[:, 1]

snapshot_preds = game_df[["play_index"]].copy()
snapshot_preds["pred_home_wp"] = cal_pred

# 3. Read the FULL original PBP (all events, no filtering)
pbp_full = pd.read_csv(pbp_path)
pbp_full.sort_values("sequence_number", inplace=True)
pbp_full.reset_index(drop=True, inplace=True)

# 4. Merge model predictions onto full PBP via sequence_number
pbp_full = pbp_full.merge(
    snapshot_preds,
    left_on="sequence_number", right_on="play_index",
    how="left",
).drop(columns=["play_index"])

# Forward-fill: plays between snapshots inherit the last model prediction.
# Plays before the first snapshot get the pre-game prior.
pbp_full["pred_home_wp"] = pbp_full["pred_home_wp"].ffill().fillna(pwp)

# 5. Add game-level context
pbp_full.insert(0, "game_id", gid)
pbp_full["prior_home_wp"] = pwp

# 6. Save
save_path = Path("data/pbp_prediction.csv")
pbp_full.to_csv(save_path, index=False)

home_win = int(game_df["home_win_final"].iloc[0])
print(f"\nSaved {len(pbp_full):,} plays → {save_path}")
print(f"  Pred range: [{pbp_full['pred_home_wp'].min():.3f}, {pbp_full['pred_home_wp'].max():.3f}]")

# 7. Preview
print(f"\n{'='*90}")
show_cols = ["sequence_number", "type_text", "text",
             "period_number", "clock_minutes", "clock_seconds",
             "home_score", "away_score", "prior_home_wp", "pred_home_wp"]
show_cols = [c for c in show_cols if c in pbp_full.columns]
pd.set_option("display.max_colwidth", 50)
print(pbp_full[show_cols].head(20).to_string(index=False))
pd.reset_option("display.max_colwidth")

IndexError: single positional indexer is out-of-bounds